# avoidZoneTest v7 (260920) — 플로깅 강도 옵션 검증 (Phase 0)

| 항목 | 내용 |
| --- | --- |
| 작성 일자 | 2026-09-20 |
| 선행 노트북 | `avoidZoneTest_v6_260827.ipynb` |
| 선행 문서 | `Ploggo_알고리즘검증_이관인계_260903.md` |
| Tmap 호출 | **0회** (좌표 연산만 수행) |
| 산출물 | `./output7_260920/` |

## 1. 검증 목적

플로깅 경로 추천에 사용자 선택 옵션 2개를 도입할 수 있는지 판정한다.

| 옵션 | 의미 | β (`ALGO_BETA`) |
| --- | --- | --- |
| **가볍게 플로깅** | 도착지까지 거의 돌아가지 않는다 | **2.0** |
| **구석구석 플로깅** | 오염 취약 지점을 더 챙기며 돌아간다 | **0.5** (현행값) |

`Score_i = α × S_i − β × D_i − γ × avoid_penalty_i` 에서 β만 바꾼다.
α = 0.5, γ = 1.0, K = 2, BF_TOP_N = 15는 고정한다.

β의 의미는 "오염도 점수 1점을 얻기 위해 몇 km까지 더 걸을 것인가"이다.

| β | 오염도 1점이 정당화하는 우회 거리 | 시속 3km 환산 |
| --- | --- | --- |
| 0.5 | 1.00 km | 20분 |
| 2.0 | 0.25 km | 5분 |

## 2. 본 노트북이 답하는 질문

**질문 1 (선결).** β를 0.5와 2.0으로 바꿨을 때 실제로 **다른 CCTV 핫스팟이 선정되는가.**

v3에서 회피 페널티 가중치 γ를 0.5~5.0으로 10배 범위에서 바꿨음에도 48쌍 중 41쌍(85%)이
완전히 동일한 결과였던 전례가 있다. 파라미터를 바꿔도 선정이 바뀌지 않는 구조가 이미 한 번
확인되었다. 여기서도 같은 위험이 있다. 후보는 2단계에서 이미 오염도 상위 15곳으로 좁혀지므로,
그 15곳 안에 경로 축에 가까운 핫스팟이 없으면 β를 아무리 키워도 선택이 바뀌지 않는다.

이 질문에서 부정이 나오면 β 조절로는 옵션을 만들 수 없으며, `BF_TOP_N` 또는 Buffer 후보
추출 조건을 손봐야 한다는 뜻이 된다.

**질문 2.** 두 옵션의 플로깅 경로 총 거리 차이가 사용자가 체감할 만한 크기인가.

**질문 3.** 가볍게 옵션이 오염도를 얼마나 희생하는가. 이것이 짧은 옵션의 실제 비용이다.

**질문 4 (부수).** 방문 순서를 점수 내림차순으로 고정한 현행 방식이 거리를 얼마나 늘리고 있는가.
현행 코드는 `# 4. K=3 방문 순서 = Score_i 내림차순`으로 지리적 위치를 고려하지 않는다.
미결 항목 `#B3 visit_order 최적화`에 해당하며, 비용 없이 함께 측정한다.

## 3. 판정 기준 (결과를 보기 전에 고정한다)

| 지표 | 성립 | 조건부 | 불성립 |
| --- | --- | --- | --- |
| 옵션 분리율 | 70% 이상 | 40~70% | 40% 미만 |
| 추정 거리 차이 중앙값 | 0.5km 이상 | 0.3~0.5km | 0.3km 미만 |
| 가볍게의 평균 S_i 손실 | 15% 미만 | 15~30% | 30% 이상 |

거리 0.5km는 시속 3km 기준 10분에 해당한다. 그 아래면 두 옵션을 구분해 제시할 근거가 약하다.

## 4. 실행 전 준비

아래 파일이 작업 폴더에 있어야 한다. v6 노트북과 동일한 위치를 가정한다.

```
./hotspots_CCTV_20260510.csv
./output/incheon_avoid_zones_v1_260601.geojson
./output6_260827/mixed_od_200.pkl
```

`.env`는 필요 없다. 외부 API를 호출하지 않는다.

In [ ]:
# 셀 1: 라이브러리 import 및 환경 점검
# v7은 외부 API를 호출하지 않는다. requests, dotenv 불필요.

import os
import ast
import json
import math
import time
import pickle
import itertools
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd

from shapely.geometry import shape, Point, LineString
from shapely.ops import unary_union
from shapely.prepared import prep
from scipy.spatial import cKDTree

print("[v7 노트북 시작 — 플로깅 강도 옵션 검증 Phase 0]")
print(f"  pandas  {pd.__version__}")
print(f"  numpy   {np.__version__}")
print(f"  작업 디렉토리: {os.getcwd()}")
print("  Tmap 호출: 없음")

In [ ]:
# 셀 2: 경로 상수 + 입력 파일 존재 점검
# 경로가 다르면 이 셀의 상수만 수정한다.

CSV_PATH     = Path("./hotspots_CCTV_20260510.csv")
GEOJSON_PATH = Path("./output/incheon_avoid_zones_v1_260601.geojson")
OD_PKL_PATH  = Path("./output6_260827/mixed_od_200.pkl")

STAMP   = "260920"
V7_ROOT = Path(f"./output7_{STAMP}")
V7_ROOT.mkdir(parents=True, exist_ok=True)

_missing = [p for p in (CSV_PATH, GEOJSON_PATH, OD_PKL_PATH) if not p.exists()]
if _missing:
    raise FileNotFoundError(
        "입력 파일이 없다:\n  " + "\n  ".join(str(p.resolve()) for p in _missing)
    )

print("[입력 파일 확인]")
for p in (CSV_PATH, GEOJSON_PATH, OD_PKL_PATH):
    print(f"  {p}  ({p.stat().st_size/1024:.0f} KB)")
print(f"[출력 폴더] {V7_ROOT.resolve()}")

In [ ]:
# 셀 3: 회피 영역 GeoJSON 로드 (v6 셀 3 이식, 분석 출력 축약)

with open(GEOJSON_PATH, "r", encoding="utf-8") as f:
    avoid_geojson = json.load(f)

avoid_polygons = []
for feature in avoid_geojson["features"]:
    props = feature["properties"]
    avoid_polygons.append({
        "geom":      shape(feature["geometry"]),
        "uqa_code":  props.get("uqa_code"),
        "sigg_name": props.get("sigg_name"),
    })

avoid_union = unary_union([p["geom"] for p in avoid_polygons])
avoid_union_prepared = prep(avoid_union)

print(f"[회피 영역 로드] 폴리곤 {len(avoid_polygons)}개")
for code in ["UQA310", "UQA320", "UQA330"]:
    n = sum(1 for p in avoid_polygons if p["uqa_code"] == code)
    print(f"  {code}: {n}개")

if len(avoid_polygons) != 147:
    print(f"  [경고] v6 기준 147개와 다르다. 데이터 판이 바뀌었는지 확인이 필요하다.")

In [ ]:
# 셀 4: 핫스팟 CSV 로드 + neighbor_200m + S_i + in_avoid_zone
# v6 셀 4 이식. 공식은 변경하지 않는다.

EARTH_RADIUS_KM = 6371.0

hotspots = pd.read_csv(CSV_PATH, encoding="utf-8-sig")


def parse_list_field(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return None
    return value


for col in ("install_years", "relocation_history"):
    if col in hotspots.columns:
        hotspots[col] = hotspots[col].apply(parse_list_field)

# neighbor_200m (cKDTree, v6와 동일 방식)
_lat_rad = np.radians(hotspots["latitude"].values)
_lon_rad = np.radians(hotspots["longitude"].values)
_xyz = np.column_stack([
    np.cos(_lat_rad) * np.cos(_lon_rad),
    np.cos(_lat_rad) * np.sin(_lon_rad),
    np.sin(_lat_rad),
])
_tree = cKDTree(_xyz)
_chord = 2 * np.sin((0.2 / EARTH_RADIUS_KM) / 2)
hotspots["neighbor_200m"] = _tree.query_ball_point(_xyz, r=_chord, return_length=True) - 1

# S_i = cctv_count + 0.5 × neighbor_200m
hotspots["S_i"] = hotspots["cctv_count"] + 0.5 * hotspots["neighbor_200m"]

# 회피 영역 내부 여부
hotspots["in_avoid_zone"] = hotspots.apply(
    lambda r: avoid_union_prepared.contains(Point(r["longitude"], r["latitude"])), axis=1
)

print(f"[핫스팟 로드] {len(hotspots):,}곳")
print(f"  S_i    평균 {hotspots['S_i'].mean():.2f}  min {hotspots['S_i'].min():.2f}  max {hotspots['S_i'].max():.2f}")
print(f"  회피 영역 내부 {hotspots['in_avoid_zone'].sum()}곳 "
      f"({hotspots['in_avoid_zone'].mean()*100:.2f}%)")

if len(hotspots) != 1535:
    print("  [경고] v6 기준 1,535곳과 다르다.")

In [ ]:
# 셀 5: 거리 유틸 + 회피 페널티 (v6 셀 5·6 이식, 수정 없음)

DETOUR_RATIO   = 1.3     # 직선 → 도보 추정 보정계수
BUFFER_R_KM    = 1.5     # v6와 동일 (군구별 분기 없음)
BUFFER_FACTOR  = 1.5     # 우회 허용치 = BUFFER_R_KM × BUFFER_FACTOR = 2.25km


def haversine_km(p_a, p_b):
    lat1, lon1 = p_a
    lat2, lon2 = p_b
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return EARTH_RADIUS_KM * 2 * math.asin(math.sqrt(a))


def walking_distance_km(p_a, p_b):
    """추정 도보 거리 (km) = Haversine × 1.3"""
    return haversine_km(p_a, p_b) * DETOUR_RATIO


def detour_distance_km(origin, hotspot, destination):
    """D_i = d(O,h) + d(h,D) - d(O,D)"""
    return max(
        walking_distance_km(origin, hotspot)
        + walking_distance_km(hotspot, destination)
        - walking_distance_km(origin, destination),
        0.0,
    )


def filter_candidates_by_ellipse_buffer(df, origin, destination, buffer_r_km=BUFFER_R_KM):
    """타원 buffer 필터: d(O,h) + d(h,D) - d(O,D) < buffer_r × 1.5"""
    d_od = walking_distance_km(origin, destination)
    threshold = buffer_r_km * BUFFER_FACTOR
    coords = df[["latitude", "longitude"]].values
    d_oh = np.array([walking_distance_km(origin, tuple(c)) for c in coords])
    d_hd = np.array([walking_distance_km(tuple(c), destination) for c in coords])
    return df[(d_oh + d_hd - d_od) < threshold].copy()


def compute_avoid_penalty(hotspot_lat, hotspot_lon, origin, destination,
                          avoid_union_param, avoid_union_prepared_param):
    """회피 영역 내부면 999, 그 외에는 O-h, h-D 직선의 회피 영역 통과 길이(km)"""
    if avoid_union_prepared_param.contains(Point(hotspot_lon, hotspot_lat)):
        return 999.0

    line_o = LineString([(origin[1], origin[0]), (hotspot_lon, hotspot_lat)])
    inter_o = line_o.intersection(avoid_union_param)
    len_o = inter_o.length if not inter_o.is_empty else 0

    line_d = LineString([(hotspot_lon, hotspot_lat), (destination[1], destination[0])])
    inter_d = line_d.intersection(avoid_union_param)
    len_d = inter_d.length if not inter_d.is_empty else 0

    return (len_o + len_d) * 111.0


print("[유틸 함수 정의 완료]")
print(f"  우회 허용치 = {BUFFER_R_KM} × {BUFFER_FACTOR} = {BUFFER_R_KM*BUFFER_FACTOR:.2f} km (핫스팟 1곳당)")

In [ ]:
# 셀 6: β 파라미터화 선정 함수 + 방문 순서 평가
# v6 셀 7·8의 선정 부분만 이식한다. Tmap 호출과 swap은 포함하지 않는다.

ALPHA     = 0.5
GAMMA     = 1.0
K         = 2
BF_TOP_N  = 15

BETA_LIGHT = 2.0   # 가볍게 플로깅
BETA_DEEP  = 0.5   # 구석구석 플로깅 (현행 ALGO_BETA)


def compute_scores(df, origin, destination, beta,
                   alpha=ALPHA, gamma=GAMMA):
    """Score_i = α·S_i - β·D_i - γ·avoid_penalty_i"""
    out = df.copy()
    out["D_i"] = out.apply(
        lambda r: detour_distance_km(origin, (r["latitude"], r["longitude"]), destination),
        axis=1,
    )
    out["avoid_penalty_i"] = out.apply(
        lambda r: compute_avoid_penalty(
            r["latitude"], r["longitude"], origin, destination,
            avoid_union, avoid_union_prepared),
        axis=1,
    )
    out["Score_i"] = alpha * out["S_i"] - beta * out["D_i"] - gamma * out["avoid_penalty_i"]
    return out


def select_hotspots(candidates_df, origin, destination, beta, k=K):
    """Buffer 후보 → S_i 상위 BF_TOP_N 풀 → Score_i 상위 K 선정.
    반환 DataFrame의 행 순서는 Score_i 내림차순이며, 이것이 현행 방문 순서다.
    """
    n = len(candidates_df)
    if n == 0:
        return None
    pool = candidates_df.nlargest(min(BF_TOP_N, n), "S_i")
    scored = compute_scores(pool, origin, destination, beta)
    return scored.nlargest(min(k, n), "Score_i")


def path_length_km(origin, destination, waypoints):
    """O → waypoints(주어진 순서) → D 의 추정 도보 거리 합"""
    pts = [origin] + list(waypoints) + [destination]
    return sum(walking_distance_km(pts[i], pts[i + 1]) for i in range(len(pts) - 1))


def best_order_length_km(origin, destination, waypoints):
    """방문 순서를 전수 탐색하여 가장 짧은 추정 거리와 그 순서를 반환한다.
    K=2면 조합 2가지, K=3이면 6가지로 비용이 사실상 0이다.
    """
    best = None
    for order in itertools.permutations(range(len(waypoints))):
        pts = [waypoints[i] for i in order]
        total = path_length_km(origin, destination, pts)
        if best is None or total < best[0]:
            best = (total, order)
    return best


print("[선정 함수 정의 완료]")
print(f"  고정: α={ALPHA}, γ={GAMMA}, K={K}, BF_TOP_N={BF_TOP_N}")
print(f"  비교: β(가볍게)={BETA_LIGHT}  vs  β(구석구석)={BETA_DEEP}")

In [ ]:
# 셀 7: v6 OD 200쌍 로드 + Buffer 후보 재계산
# pkl에 저장된 candidates를 쓰지 않고 현재 hotspots에서 다시 계산한다.
# 컬럼 정합 문제를 원천 차단하고, 동시에 v6 재현성을 점검한다.

with open(OD_PKL_PATH, "rb") as f:
    od_list = pickle.load(f)

print(f"[OD 표본 로드] {len(od_list)}쌍")
print(f"  인접군 {sum(1 for o in od_list if o.get('group') == 'adjacent')}쌍")
print(f"  분리군 {sum(1 for o in od_list if o.get('group') == 'separated')}쌍")

t0 = time.time()
n_mismatch = 0
for od in od_list:
    cands = filter_candidates_by_ellipse_buffer(hotspots, od["O"], od["D"])
    if "n_candidates" in od and od["n_candidates"] != len(cands):
        n_mismatch += 1
    od["candidates_v7"] = cands

print(f"\n[Buffer 후보 재계산] 소요 {time.time() - t0:.1f}초")
print(f"  평균 후보 수 {np.mean([len(o['candidates_v7']) for o in od_list]):.1f}곳")
print(f"  v6 저장값과 불일치: {n_mismatch}쌍")
if n_mismatch:
    print("  [경고] 불일치가 있다. 핫스팟 CSV 판 또는 Buffer 상수가 v6와 다른지 확인해야 한다.")

In [ ]:
# 셀 8: 메인 비교 루프 — β=2.0(가볍게) vs β=0.5(구석구석)
# Tmap 호출 없음. 좌표 연산만 수행한다.

rows = []
t0 = time.time()

for idx, od in enumerate(od_list):
    origin, destination = od["O"], od["D"]
    cands = od["candidates_v7"]

    rec = {
        "od_id":        od.get("od_id"),
        "group":        od.get("group"),
        "district":     od.get("district"),
        "direct_km":    od.get("direct_km", haversine_km(origin, destination)),
        "chord_m":      od.get("chord_m", 0.0),
        "n_candidates": len(cands),
        "walk_od_km":   walking_distance_km(origin, destination),
    }

    if len(cands) < K:
        rec["status"] = "후보부족"
        rows.append(rec)
        continue

    sel = {}
    for tag, beta in (("light", BETA_LIGHT), ("deep", BETA_DEEP)):
        s = select_hotspots(cands, origin, destination, beta)
        wps = [(r["latitude"], r["longitude"]) for _, r in s.iterrows()]

        est_score_order = path_length_km(origin, destination, wps)
        est_best_order, order = best_order_length_km(origin, destination, wps)

        sel[tag] = set(s["hotspot_id"])
        rec[f"{tag}_ids"]        = "|".join(str(x) for x in s["hotspot_id"])
        rec[f"{tag}_mean_Si"]    = float(s["S_i"].mean())
        rec[f"{tag}_sum_Di"]     = float(s["D_i"].sum())
        rec[f"{tag}_est_km"]     = est_score_order        # 현행 방식 (점수 순 방문)
        rec[f"{tag}_est_best_km"] = est_best_order        # 순서 최적화 적용 시
        rec[f"{tag}_order_loss_km"] = est_score_order - est_best_order

    # 옵션 분리 여부
    rec["separated_option"] = (sel["light"] != sel["deep"])
    rec["n_common"]  = len(sel["light"] & sel["deep"])
    rec["est_gap_km"] = rec["deep_est_km"] - rec["light_est_km"]
    rec["Si_loss_pct"] = (
        (rec["deep_mean_Si"] - rec["light_mean_Si"]) / rec["deep_mean_Si"] * 100
        if rec["deep_mean_Si"] > 0 else 0.0
    )
    rec["status"] = "OK"
    rows.append(rec)

    if (idx + 1) % 50 == 0:
        print(f"  진행 {idx + 1}/{len(od_list)}")

df = pd.DataFrame(rows)
print(f"\n[비교 완료] 소요 {time.time() - t0:.1f}초, {len(df)}행")
print(f"  정상 {int((df['status'] == 'OK').sum())}쌍 / 후보부족 {int((df['status'] == '후보부족').sum())}쌍")

In [ ]:
# 셀 9: 집계 — 질문 1~4에 대한 답

ok = df[df["status"] == "OK"].copy()

print("=" * 78)
print("[질문 1] 옵션 분리율 — β를 바꾸면 실제로 다른 핫스팟이 선정되는가")
print("=" * 78)

def sep_rate(sub, label):
    if len(sub) == 0:
        print(f"  {label:<10} 표본 없음")
        return
    r = sub["separated_option"].mean() * 100
    print(f"  {label:<10} {sub['separated_option'].sum():>3}/{len(sub):<3}  {r:5.1f}%   "
          f"(2곳 모두 동일 {int((sub['n_common'] == K).sum())}쌍)")

sep_rate(ok, "전체")
sep_rate(ok[ok["group"] == "adjacent"], "인접군")
sep_rate(ok[ok["group"] == "separated"], "분리군")

print()
print("=" * 78)
print("[질문 2] 두 옵션의 추정 플로깅 경로 거리 차이")
print("=" * 78)
print(f"  가볍게   평균 {ok['light_est_km'].mean():.2f}km  중앙값 {ok['light_est_km'].median():.2f}km")
print(f"  구석구석 평균 {ok['deep_est_km'].mean():.2f}km  중앙값 {ok['deep_est_km'].median():.2f}km")
print(f"  차이     평균 {ok['est_gap_km'].mean():.2f}km  중앙값 {ok['est_gap_km'].median():.2f}km")
print(f"           시속 3km 환산 중앙값 {ok['est_gap_km'].median()/3*60:.0f}분")
print(f"  차이 0.5km 이상: {int((ok['est_gap_km'] >= 0.5).sum())}/{len(ok)}쌍 "
      f"({(ok['est_gap_km'] >= 0.5).mean()*100:.1f}%)")
print(f"  역전(가볍게가 더 김): {int((ok['est_gap_km'] < 0).sum())}쌍")

print()
print("=" * 78)
print("[질문 3] 가볍게 옵션의 오염도 손실")
print("=" * 78)
print(f"  선정 핫스팟 평균 S_i — 구석구석 {ok['deep_mean_Si'].mean():.2f} / "
      f"가볍게 {ok['light_mean_Si'].mean():.2f}")
print(f"  손실률 평균 {ok['Si_loss_pct'].mean():.1f}%  중앙값 {ok['Si_loss_pct'].median():.1f}%")
print(f"  평균 우회 합계 D_i — 구석구석 {ok['deep_sum_Di'].mean():.2f}km / "
      f"가볍게 {ok['light_sum_Di'].mean():.2f}km")

print()
print("=" * 78)
print("[질문 4] 방문 순서를 점수 순으로 고정해 생기는 거리 손실 (미결 항목 #B3)")
print("=" * 78)
for tag, label in (("light", "가볍게"), ("deep", "구석구석")):
    loss = ok[f"{tag}_order_loss_km"]
    print(f"  {label:<8} 평균 +{loss.mean():.2f}km  중앙값 +{loss.median():.2f}km  "
          f"최대 +{loss.max():.2f}km  (순서가 최적이 아닌 OD {int((loss > 0.001).sum())}쌍)")

In [ ]:
# 셀 10: 판정 + 산출물 저장

csv_path = V7_ROOT / f"beta_option_compare_{STAMP}.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

sep_pct  = ok["separated_option"].mean() * 100
gap_med  = ok["est_gap_km"].median()
si_loss  = ok["Si_loss_pct"].mean()


def verdict(value, good, mid, higher_is_better=True):
    if higher_is_better:
        return "성립" if value >= good else ("조건부" if value >= mid else "불성립")
    return "성립" if value < good else ("조건부" if value < mid else "불성립")


print("=" * 78)
print("[판정 — 셀 0의 사전 기준 적용]")
print("=" * 78)
print(f"  옵션 분리율        {sep_pct:5.1f}%   → {verdict(sep_pct, 70, 40)}")
print(f"  거리 차이 중앙값   {gap_med:5.2f}km  → {verdict(gap_med, 0.5, 0.3)}")
print(f"  가볍게 S_i 손실    {si_loss:5.1f}%   → {verdict(si_loss, 15, 30, higher_is_better=False)}")
print()
print("  세 항목이 모두 '성립'이면 Phase 1(Tmap 실측)으로 진행한다.")
print("  분리율이 '불성립'이면 β 조절로는 옵션을 만들 수 없다는 뜻이며,")
print("  BF_TOP_N(=15) 또는 Buffer 우회 허용치(=2.25km)를 조절 대상으로 바꿔야 한다.")
print()
print(f"[저장] {csv_path.resolve()}")

In [ ]:
# 셀 11: 분리율이 낮게 나왔을 때의 원인 진단
# β를 바꿔도 선정이 안 바뀌는 이유를 정량적으로 확인한다.
#
# 두 핫스팟 i, j의 Score 대소는 (α·ΔS_i - β·ΔD_i - γ·ΔP_i)의 부호로 정해진다.
# 따라서 순위가 뒤집히는 β 값은 다음 하나뿐이다.
#
#     β* = (α·ΔS - γ·ΔP) / ΔD
#
# β*가 0.5와 2.0 사이에 있는 쌍이 하나도 없으면, 두 옵션의 선정 결과는
# 원리적으로 동일할 수밖에 없다. 풀 안에 그런 쌍이 몇 개 있는지 센다.

BETA_LO, BETA_HI = min(BETA_DEEP, BETA_LIGHT), max(BETA_DEEP, BETA_LIGHT)

diag = []
for od in od_list:
    cands = od["candidates_v7"]
    if len(cands) < K:
        continue
    pool = cands.nlargest(min(BF_TOP_N, len(cands)), "S_i")
    scored = compute_scores(pool, od["O"], od["D"], beta=BETA_DEEP)

    S = scored["S_i"].values
    D = scored["D_i"].values
    P = scored["avoid_penalty_i"].values

    n_flip = 0
    n_pair = 0
    for a, b in itertools.combinations(range(len(scored)), 2):
        dD = D[a] - D[b]
        if abs(dD) < 1e-9:
            continue
        n_pair += 1
        beta_star = (ALPHA * (S[a] - S[b]) - GAMMA * (P[a] - P[b])) / dD
        if BETA_LO < beta_star < BETA_HI:
            n_flip += 1

    diag.append({
        "od_id":        od.get("od_id"),
        "group":        od.get("group"),
        "n_candidates": len(cands),
        "pool_size":    len(scored),
        "n_pairs":      n_pair,
        "n_flip_pairs": n_flip,
        "flip_ratio":   n_flip / n_pair if n_pair else 0.0,
        "pool_Di_range": float(D.max() - D.min()),
        "pool_Si_range": float(S.max() - S.min()),
    })

dg = pd.DataFrame(diag)

print("[진단] β를 0.5에서 2.0으로 올릴 때 순위가 뒤집힐 수 있는 핫스팟 쌍")
print(f"  OD당 평균 {dg['n_flip_pairs'].mean():.1f}쌍 / 전체 비교쌍 {dg['n_pairs'].mean():.0f}쌍")
print(f"  뒤집힘 쌍이 0인 OD: {int((dg['n_flip_pairs'] == 0).sum())}/{len(dg)}쌍")
print(f"  풀 내 D_i 폭 평균 {dg['pool_Di_range'].mean():.2f}km, S_i 폭 평균 {dg['pool_Si_range'].mean():.2f}점")
print()
print("  해석 기준")
print(f"    뒤집힘 쌍 0 = β 조절로 선정을 바꿀 수 없는 OD. 이 비율이 높으면")
print(f"    β가 아니라 BF_TOP_N(={BF_TOP_N}) 또는 Buffer 우회 허용치를 조절 대상으로 바꿔야 한다.")
print(f"    참고로 S_i가 1점 차이나는 두 핫스팟이 뒤집히려면, 우회거리 차이가")
print(f"    {ALPHA/BETA_HI:.2f}km ~ {ALPHA/BETA_LO:.2f}km 구간에 있어야 한다 (회피 페널티 동일 가정).")

dg.to_csv(V7_ROOT / f"pool_diagnosis_{STAMP}.csv", index=False, encoding="utf-8-sig")
print(f"\n[저장] {(V7_ROOT / f'pool_diagnosis_{STAMP}.csv').resolve()}")